# Member 4 — DenseNet121 Transfer Learning

## Comparative Analysis of Deep Learning Architectures for Multi-Class Brain Tumour MRI Classification

**SE4050 – Deep Learning 2026**

This notebook implements a DenseNet121 transfer-learning model for classifying brain tumour MRI images into four classes: **glioma**, **meningioma**, **notumor**, and **pituitary**.

### Architectural Context & Scientific Comparison
DenseNet121 introduces **dense connectivity** across convolutional layers (Huang et al., 2017). While ResNet combines features through element-wise addition ($x + f(x)$), DenseNet concatenates all preceding feature maps ($[x_0, x_1, \dots, x_{l-1}]$). This architecture promotes maximum feature reuse, mitigates the vanishing-gradient problem, and substantially reduces parameter count relative to standard deep convolutional stacks (7.0M backbone parameters vs. 14.7M in VGG16 and 23.6M in ResNet50).

---

### Important Scientific Limitations
1. **Transfer Learning Limitation:** DenseNet121 starts from ImageNet pretrained weights, whereas Member 1's Custom CNN was trained from scratch. The comparison evaluates practical transfer-learning paradigms against custom from-scratch training, not an isolated architecture-only benchmark.
2. **Patient-Level Independence Limitation:** Patient-level independence could not be independently verified from the available dataset structure and identifiers. All analyses report image-level generalization.

---
## Phase 1 — Environment Verification and DenseNet121 Preprocessing Pipeline

**Goal:** Verify environment reproducibility, validate Member 1's frozen split manifests via SHA-256 checksums, and construct the DenseNet121-specific `tf.data` input pipeline.

### Phase 1 Tasks
1. Record complete runtime environment (Python, TensorFlow, Keras, NumPy, Pandas, scikit-learn, GPU, determinism).
2. Verify SHA-256 checksums for all three frozen split CSV files (`train.csv`, `val.csv`, `test.csv`).
3. Download exact Kaggle dataset version 1 (`masoudnickparvar/brain-tumor-mri-dataset/versions/1`).
4. Verify frozen sample counts: Train = 4,353, Validation = 1,089, Test = 1,311.
5. Verify frozen class mapping: `glioma: 0`, `meningioma: 1`, `notumor: 2`, `pituitary: 3`.
6. Verify that every referenced image file exists on disk.
7. Build DenseNet121-specific `tf.data` pipeline using `keras.applications.densenet.preprocess_input` (NO `Rescaling(1./255)`).
8. Apply common mild augmentation to training data only (rotation $\approx 0.03$, zoom $\approx \pm 0.08$, translation $\approx 0.05$).
9. Verify batch shapes, label integrity, and unshuffled validation stream.
10. Verify held-out test manifest without creating an evaluation dataset.
11. Save Phase 1 configuration and environment manifest.

### 1.1 — Install Dependencies and Imports

In [1]:
# Install kagglehub if not already present in environment
!pip install -q kagglehub

import os
import sys
import json
import hashlib
import time
import random
import platform
import shutil
import subprocess
import gc

import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import sklearn
import kagglehub
import matplotlib
import matplotlib.pyplot as plt

# Architecture-specific preprocessor for DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess_input

print("All imports completed successfully.")

All imports completed successfully.
